# 🚕 movio — Tanglish TTS Fine-Tuning on Kaggle

**End-to-end pipeline:** download data → prepare corpus → build dataset → fine-tune IndicF5 → merge/export → evaluate → demo inference.

Everything runs top-to-bottom in this single notebook. **Nothing needs to be trained locally.**

---
## ⚙️ Before you run anything — Kaggle UI checklist

| # | Setting | Where | Value |
|---|---|---|---|
| 1 | Accelerator | Settings (right panel) | **GPU T4 x2** (or P100) |
| 2 | Internet | Settings | **ON** (requires verified phone number on your Kaggle account) |
| 3 | Persistence | Settings | ON (optional, keeps `/kaggle/working` between sessions) |
| 4 | HF token | Add-ons → Secrets | Add a secret named **`HF_TOKEN`** |

## 🔑 Hugging Face access (do this days before!)

Both resources are **gated** — approval takes 24–48 h:
1. Request access to the model: https://huggingface.co/ai4bharat/IndicF5
2. Request access to the dataset: https://huggingface.co/datasets/ai4bharat/indicvoices_r
3. Create a read token: https://huggingface.co/settings/tokens → put it in the `HF_TOKEN` Kaggle secret.

## 📦 Your own Tanglish recordings (recommended)

The v2 quality boost comes from real Chennai-region Tanglish audio. If you have recordings:
1. Shape them as: `audio/*.wav` + `transcripts.tsv` (one `filename<TAB>transcript` pair per line).
2. Upload as a **private Kaggle Dataset** named e.g. `yourname/tanglish-recordings`.
3. Click **+ Add Input** in this notebook and attach that dataset.

If you skip this, the pipeline still works using only indicvoices_r Tamil (base adaptation, no switch-point prosody boost).


In [ ]:
# ── Central configuration — edit these paths only ──
REPO_URL = "https://github.com/YOUR_USERNAME/movio.git"   # ← change me!
REPO_DIR = "/kaggle/working/movio"

# Optional: your private Kaggle Dataset with custom recordings.
# Set to None to train on indicvoices_r Tamil only.
CUSTOM_DATA_INPUT = "/kaggle/input/tanglish-recordings"   # or None

WORK = "/kaggle/working"
RAW_DIR = f"{WORK}/raw"
DATA_DIR = f"{WORK}/data"
FT_OUT = f"{WORK}/f5tts_out"
MERGED_DIR = f"{WORK}/indicf5_tanglish_merged"
BASE_MODEL = "ai4bharat/IndicF5"

import os
for d in (WORK, RAW_DIR, DATA_DIR):
    os.makedirs(d, exist_ok=True)
print("config OK")


## 1 · Environment check


In [ ]:
import subprocess, sys

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout[-3000:] if r.stdout else '', r.stderr[-1500:] if r.returncode else '')
    return r.returncode

try:
    import torch
except ImportError:
    sh(f'{sys.executable} -m pip install -q torch --index-url https://download.pytorch.org/whl/cu121')
    import torch

assert torch.cuda.is_available(), (
    '\U0001F6A8 No CUDA! Go to Settings → Accelerator → GPU T4 x2 / P100, then restart.')
ngpu = torch.cuda.device_count()
for i in range(ngpu):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name} ({p.total_memory/1e9:.1f} GB)')
print('torch', torch.__version__)


## 2 · Hugging Face authentication

Reads the `HF_TOKEN` secret you added via **Add-ons → Secrets**.


In [ ]:
import os

if not (os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')):
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']

from huggingface_hub import whoami
print('authenticated as:', whoami()['name'])


## 3 · Get the movio repo

Two options — the cell picks automatically:
- **Option A:** the repo is attached as a Kaggle Dataset input at `/kaggle/input/movio`
  (upload your repo via kaggle datasets create, good for private code), or
- **Option B:** `git clone` from `REPO_URL`. For a **private** GitHub repo, add a second
  Kaggle secret `GITHUB_TOKEN` and we clone with it.

If you already cloned in a previous session, the existing copy is reused (`git pull`).


In [ ]:
import os, shutil, subprocess

if os.path.exists('/kaggle/input/movio') and not os.path.exists(REPO_DIR):
    print('Option A: copying attached dataset input…')
    shutil.copytree('/kaggle/input/movio', REPO_DIR)
elif not os.path.exists(REPO_DIR):
    print('Option B: git clone…')
    url = REPO_URL
    try:
        gh_token = UserSecretsClient().get_secret('GITHUB_TOKEN')
        url = url.replace('https://', f'https://{gh_token}@')
        print('(using GITHUB_TOKEN for private repo)')
    except Exception:
        pass
    subprocess.run(['git', 'clone', url, REPO_DIR], check=True)
else:
    print('repo exists — pulling latest')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True)

%cd {REPO_DIR}
!ls


## 4 · Install dependencies


In [ ]:
!pip install -q -r requirements.txt
!pip install -q jiwer peft datasets soundfile librosa psutil websockets
import transformers, accelerate
print('transformers', transformers.__version__, '| accelerate', accelerate.__version__)


## 5 · Download indicvoices_r (Tamil subset)

CC-BY-4.0, ~commercial-clean. We pull **only the Tamil split** (~24 GB full).
The script is **disk-budget aware** — it downloads as much as fits (up to
free disk minus 5 GB headroom, capped at 15 GB by default) and stops cleanly.
On a standard Kaggle instance you'll get ~15 GB of Tamil audio, which is
plenty for fine-tuning.

⏱ Expect 5–20 min depending on Kaggle network.

In [ ]:
!python training/scripts/01_download_data.py --out {RAW_DIR} --language Tamil

import pathlib
n_parquet = len(list(pathlib.Path(RAW_DIR).rglob('*.parquet')))
n_wav = len(list(pathlib.Path(RAW_DIR).rglob('*.wav')))
print(f'\nparquet files: {n_parquet} | standalone wav files: {n_wav}')
assert n_parquet > 0 or n_wav > 0, 'Download produced no data — check HF_TOKEN / gated access approval.'

## 6 · Attach your custom Tanglish recordings (optional but recommended)

Expected layout inside your Kaggle Dataset:
```
audio/
  rec_0001.wav
  rec_0002.wav
transcripts.tsv      # rec_0001.wav<TAB>Unga pickup location enga?
```


In [ ]:
import os

if CUSTOM_DATA_INPUT and os.path.isdir(CUSTOM_DATA_INPUT):
    # merge custom audio+tsv into RAW_DIR so step 7 processes both sources
    import shutil, pathlib
    dst = pathlib.Path(RAW_DIR) / 'custom'
    dst.mkdir(parents=True, exist_ok=True)
    for src in pathlib.Path(CUSTOM_DATA_INPUT).rglob('*'):
        if src.suffix.lower() in ('.wav', '.tsv'):
            shutil.copy(src, dst / src.name)
    n = len(list(dst.glob('*.wav')))
    print(f'copied {n} custom recordings')
else:
    print('No custom dataset attached — continuing with indicvoices_r only.')


## 7 · Prepare the corpus

Resamples everything to 24 kHz mono, trims silence, filters by duration
(1–12 s) and SNR (≥12 dB), dedupes, splits train/val/test 94/3/3, and writes
`ref_pool.csv` (200 clips used as zero-shot prompts during training).

⏱ ~5–15 min depending on corpus size. Watch the final line: **hours of usable audio**.


In [ ]:
!python training/scripts/02_prepare_corpus.py \
    --raw-dir {RAW_DIR} \
    --out {DATA_DIR} \
    --sample-rate 24000 \
    --min-dur 1.0 --max-dur 12.0 --min-snr 12.0 \
    --delete-parquets

import pandas as pd
train = pd.read_csv(f'{DATA_DIR}/train.csv')
train.sample(5)

## 8 · Build the training dataset

Produces the Arrow dataset (text + target audio + sampled `(ref_audio, ref_text)`
prompt pairs — exactly IndicF5's zero-shot conditioning) plus F5-TTS-format `metadata.csv`.


In [ ]:
!python training/scripts/03_build_dataset.py --data-dir {DATA_DIR}


## 9 · Fine-tune (LoRA via the F5-TTS trainer)

Clones SWivid/F5-TTS (the native trainer family for IndicF5's architecture),
symlinks our audio into its expected layout, generates a finetune config, and
launches `f5-tts finetune`.

**Budget guidance (single T4):**

| Audio after step 7 | Epochs | Approx time |
|---|---|---|
| 5 h | 4–6 | ~3–5 h |
| 20 h | 4–6 | ~10–16 h ⚠️ may exceed one Kaggle session (12 h) — use Save Version + resume |
| 50–100 h | 2–3 | split across multiple sessions |

> 💡 Kaggle sessions cap at 12 h (9 h without phone verification). For big corpora,
> reduce epochs here and re-run the notebook resuming from checkpoints.

⏱ This is the long cell — keep the browser tab open.


In [ ]:
EPOCHS = 1          # 4ep overfit badly (-0.198), 2ep still -0.152; try 1ep with lower LR
BATCH_SIZE = 2      # T4-safe; raise to 4 on P100/A100

!python training/scripts/04_train_lora.py --path f5tts \
    --data-dir {DATA_DIR} --out {FT_OUT} \
    --epochs {EPOCHS} --batch-size {BATCH_SIZE}

## 10 · Locate the trained checkpoint


In [ ]:
import pathlib
ckpts = sorted(pathlib.Path(FT_OUT).rglob('*.pt')) + sorted(pathlib.Path(FT_OUT).rglob('*.safetensors'))
for c in ckpts:
    print(f'{c.stat().st_size/1e6:9.1f} MB  {c}')
assert ckpts, 'No checkpoint found — did step 9 complete?'

# Find the best checkpoint (prefer model_last.pt)
CKPT_PATH = None
for c in ckpts:
    if c.name == 'model_last.pt':
        CKPT_PATH = str(c)
        break
if not CKPT_PATH:
    CKPT_PATH = str(ckpts[-1])
print('\nusing:', CKPT_PATH)

## 11 · Export fine-tuned checkpoint

Strips the EMA prefix from the F5-TTS .pt checkpoint and packages it alongside
the vocab into a self-contained bundle that the inference stack can load directly.

In [ ]:
import shutil, os, pathlib

# Free disk: remove only pip cache (keep HF cache for base model comparison later)
for cleanup in [pathlib.Path('/root/.cache/pip')]:
    if cleanup.exists():
        shutil.rmtree(cleanup, ignore_errors=True)
        print(f'freed: {cleanup}')

# Remove intermediate training checkpoints (keep only model_last.pt)
F5TTS_DIR = f'{FT_OUT}/F5-TTS'
ckpt_dir = pathlib.Path(F5TTS_DIR) / 'ckpts' / 'movio_tanglish'
if ckpt_dir.exists():
    for f in list(ckpt_dir.glob('model_*.pt')):
        if f.name != 'model_last.pt':
            f.unlink(missing_ok=True)
            print(f'removed intermediate: {f.name}')
    for f in list(ckpt_dir.glob('pretrained_*.safetensors')):
        f.unlink(missing_ok=True)
        print(f'removed pretrained safetensors: {f.name}')

free_gb = shutil.disk_usage('/kaggle/working').free / (1024**3)
print(f'disk free before export: {free_gb:.1f} GB')

!python training/scripts/05_merge_export.py \
    --ckpt {CKPT_PATH} \
    --f5tts-dir {F5TTS_DIR} \
    --out {MERGED_DIR}

print('\nexported model contents:')
!ls -lh {MERGED_DIR}

## 12 · Objective evaluation — promotion gate

| Metric | Gate |
|---|---|
| UTMOS | ≥ base − 0.05 |
| WER (Whisper-Tamil round-trip) | ≤ base + 2% absolute |

Run for **both** base and fine-tuned models, then compare.


In [ ]:
N_EVAL = 50   # raise to 100+ if time allows

!python training/scripts/06_evaluate.py \
    --model {MERGED_DIR} \
    --f5tts-dir {F5TTS_DIR} \
    --test {DATA_DIR}/test.csv \
    --ref-pool {DATA_DIR}/ref_pool.csv \
    --num-samples {N_EVAL} --skip-wer \
    --out {WORK}/eval_finetuned.json

In [ ]:
# baseline comparison (same prompts, base weights)
!python training/scripts/06_evaluate.py \
    --model base \
    --f5tts-dir {F5TTS_DIR} \
    --test {DATA_DIR}/test.csv \
    --ref-pool {DATA_DIR}/ref_pool.csv \
    --num-samples {N_EVAL} --skip-wer \
    --out {WORK}/eval_base.json

In [ ]:
import json
ft = json.load(open(f'{WORK}/eval_finetuned.json'))
base = json.load(open(f'{WORK}/eval_base.json'))
u_ft, u_b = ft['utmos']['mean'], base['utmos']['mean']
print(f'UTMOS  base={u_b:.3f}  finetuned={u_ft:.3f}  delta={u_ft-u_b:+.3f}')
verdict = '✅ PASS' if u_ft >= u_b - 0.05 else '❌ FAIL — do not promote; adjust epochs/data'
print('gate:', verdict)


## 13 · Listen: base vs fine-tuned (A/B demo)

Synthesizes the same Tanglish sentences with both models and saves WAVs to
the notebook output — play them right here with the audio widget.


In [ ]:
import sys, csv as _csv, numpy as np, soundfile as sf, torch, pathlib
import IPython.display as ipd

if f'{F5TTS_DIR}/src' not in sys.path:
    sys.path.insert(0, f'{F5TTS_DIR}/src')

# Patch __init__.py so "from f5_tts.model.backbones.dit import DiT" doesn't
# trigger Trainer → dataset.py → datasets 5.0.0 → huggingface_hub crash
init_py = pathlib.Path(f'{F5TTS_DIR}/src/f5_tts/model/__init__.py')
init_src = init_py.read_text()
if 'try:' not in init_src and 'from f5_tts.model.trainer import Trainer' in init_src:
    init_src = init_src.replace(
        'from f5_tts.model.trainer import Trainer',
        'try:\n    from f5_tts.model.trainer import Trainer\nexcept ImportError:\n    Trainer = None',
    )
    init_py.write_text(init_src)
    print('patched model/__init__.py')

# Clear cached f5_tts modules so the patched version loads
for mod_name in list(sys.modules):
    if mod_name.startswith('f5_tts'):
        del sys.modules[mod_name]

from f5_tts.model.backbones.dit import DiT
from f5_tts.model.cfm import CFM
from f5_tts.infer.utils_infer import (
    load_vocoder, load_checkpoint, infer_process,
    preprocess_ref_audio_text, get_tokenizer,
)

DEMO = [
    ('உங்கள் pickup location எங்கே?', 'ta_en_mix'),
    ('Unga OTP enna? Please share it.', 'ta_roman'),
    ('Your cab will arrive in 10 minutes.', 'en'),
    ('உங்கள் ஓட்டுநர் Chennai Central-ல இருக்கார்.', 'proper_noun_mix'),
]

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ref_row = next(iter(_csv.DictReader(open(f'{DATA_DIR}/ref_pool.csv'))))
ref_audio_path, ref_text = ref_row['audio'], ref_row['text']

model_cfg = dict(dim=1024, depth=22, heads=16, ff_mult=2, text_dim=512, conv_layers=4)
vocoder = load_vocoder(vocoder_name='vocos', is_local=False, device=device)

# Resolve base checkpoint — may have been deleted to save disk; re-download if needed
base_ckpt = f'{F5TTS_DIR}/ckpts/movio_tanglish/pretrained_model_1250000.safetensors'
if not pathlib.Path(base_ckpt).exists():
    from huggingface_hub import hf_hub_download
    base_ckpt = hf_hub_download('SWivid/F5-TTS', 'F5TTS_v1_Base/model_1250000.safetensors')
    print(f'downloaded base checkpoint: {base_ckpt}')

models = {}
for name, ckpt, vocab, use_ema in [
    ('base',
     base_ckpt,
     f'{F5TTS_DIR}/src/f5_tts/infer/examples/vocab.txt',
     True),
    ('finetuned',
     f'{MERGED_DIR}/model.pt',
     f'{MERGED_DIR}/vocab.txt',
     False),
]:
    vocab_char_map, vocab_size = get_tokenizer(vocab, 'custom')
    m = CFM(
        transformer=DiT(**model_cfg, text_num_embeds=vocab_size, mel_dim=100),
        mel_spec_kwargs=dict(n_fft=1024, hop_length=256, win_length=1024,
                             n_mel_channels=100, target_sample_rate=24000, mel_spec_type='vocos'),
        vocab_char_map=vocab_char_map,
    )
    load_checkpoint(m, ckpt, device=device, use_ema=use_ema)
    models[name] = m.to(device).eval()

ref_audio, ref_text_p = preprocess_ref_audio_text(ref_audio_path, ref_text)

for text, tag in DEMO:
    print(f'\n[{tag}] {text}')
    for name, m in models.items():
        audio, _, _ = infer_process(ref_audio, ref_text_p, text, m, vocoder, device=device)
        path = f'{WORK}/demo_{name}_{tag}.wav'
        sf.write(path, audio, 24000)
        print(name)
        ipd.display(ipd.Audio(path))

## 14 · Package & save

Everything in `/kaggle/working` is kept when you click **Save Version**
(choose *Save & Run All* for a reproducible artifact).


In [ ]:
import shutil

shutil.make_archive(f'{WORK}/movio_ft_model', 'zip', MERGED_DIR)
shutil.make_archive(f'{WORK}/movio_eval_reports', 'zip', WORK) if False else None

!ls -lh {WORK}/*.zip {WORK}/eval_*.json
print('''
Done ✅  Next steps:
 1. Download movio_ft_model.zip (Save Version → Output files)
 2. Unzip next to your serving host, e.g. /models/indicf5_tanglish_merged
 3. In config/settings.yaml set:  stage_c.model_id: /models/indicf5_tanglish_merged
 4. Restart the server — no code changes needed.
''')


## 15 · Upload artifacts to Weights & Biases

Add a Kaggle secret named **`WANDB_API_KEY`** before running this cell.

In [ ]:
import os, json, glob, pathlib
import subprocess, sys

# ── get W&B key ──────────────────────────────────────────────────────────────
if not os.environ.get('WANDB_API_KEY'):
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wandb'], check=True)
import wandb

# ── load eval results ─────────────────────────────────────────────────────────
ft   = json.load(open(f'{WORK}/eval_finetuned.json'))
base = json.load(open(f'{WORK}/eval_base.json'))
u_ft, u_b = ft['utmos']['mean'], base['utmos']['mean']
passed = u_ft >= u_b - 0.05

# ── init run ──────────────────────────────────────────────────────────────────
run = wandb.init(
    project='movio-tts',
    name=f'tanglish-ft-{EPOCHS}ep',
    config={
        'epochs':        EPOCHS,
        'batch_size':    BATCH_SIZE,
        'learning_rate': 5e-6,
        'dataset_hours': ft.get('n_samples', 50),
        'base_model':    'SWivid/F5-TTS:F5TTS_v1_Base',
        'tokenizer':     'char',
    },
    tags=['f5tts', 'tanglish', 'tamil'],
)

# ── log scalar metrics ────────────────────────────────────────────────────────
wandb.log({
    'utmos/finetuned': u_ft,
    'utmos/base':      u_b,
    'utmos/delta':     u_ft - u_b,
    'promotion_gate':  int(passed),
    'n_eval_samples':  ft['n_samples'],
})
print(f'UTMOS  base={u_b:.3f}  finetuned={u_ft:.3f}  delta={u_ft-u_b:+.3f}')
print('gate:', '✅ PASS' if passed else '❌ FAIL')

# ── upload model artifact ─────────────────────────────────────────────────────
model_artifact = wandb.Artifact(
    name='movio-tanglish-f5tts',
    type='model',
    description='Fine-tuned F5-TTS checkpoint for Tamil/Tanglish TTS',
    metadata={
        'epochs':     EPOCHS,
        'utmos_ft':   u_ft,
        'utmos_base': u_b,
        'passed':     passed,
    },
)
model_artifact.add_dir(MERGED_DIR, name='model')
run.log_artifact(model_artifact)
print('model artifact uploaded')

# ── upload eval JSON as artifact ──────────────────────────────────────────────
eval_artifact = wandb.Artifact('movio-tanglish-eval', type='evaluation')
eval_artifact.add_file(f'{WORK}/eval_finetuned.json')
eval_artifact.add_file(f'{WORK}/eval_base.json')
run.log_artifact(eval_artifact)
print('eval artifact uploaded')

# ── upload demo WAVs as audio ─────────────────────────────────────────────────
audio_logs = {}
for wav in sorted(glob.glob(f'{WORK}/demo_*.wav')):
    name = pathlib.Path(wav).stem          # e.g. demo_finetuned_ta_en_mix
    audio_logs[name] = wandb.Audio(wav, sample_rate=24000, caption=name)
if audio_logs:
    wandb.log({'demo_audio': list(audio_logs.values())})
    print(f'logged {len(audio_logs)} demo WAVs')

# ── upload training loss curve from F5-TTS log (if present) ──────────────────
log_dir = pathlib.Path(f'{F5TTS_DIR}/runs')
if log_dir.exists():
    tb_files = list(log_dir.rglob('events.out.tfevents.*'))
    if tb_files:
        tb_artifact = wandb.Artifact('movio-tanglish-tb-logs', type='tensorboard')
        for f in tb_files:
            tb_artifact.add_file(str(f))
        run.log_artifact(tb_artifact)
        print(f'uploaded {len(tb_files)} tensorboard event file(s)')

run.finish()
print(f'\nW&B run: {run.url}')